In [1]:
import pandas as pd
from ITER_DBSCAN import ITER_DBSCAN
from evaluation import EvaluateDataset

2026-02-19 14:36:32.300946: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
filepath = "review_dengan_intent.csv"
df = pd.read_csv(filepath)
df.head(5)

,userName,score,content,at,intent
0,Nabila Livia,5,sukaa,2026-02-18 10:36:04,Praise & Gratitude
1,Antok Sumawan,5,ya bagus,2026-02-18 10:32:49,Praise & Gratitude
2,Khayyira Ira,5,aku suka banget sama tiktok ini,2026-02-18 10:32:45,Praise & Gratitude
3,Aksay Subang,1,akun gua entah kenapa di Ben padalah gua kaga ...,2026-02-18 10:31:33,Account Issue
4,kenzo zildane alvaro,1,tolong diperbaiki,2026-02-18 10:30:44,General Request/Complaint


In [3]:
print('Before: ', len(df))
df = df.dropna()
print('After: ', len(df))
df = df.reset_index()
del df['index']
df.intent.value_counts()

Before:  1000
After:  1000


intent
Unlabeled/Noise              277
Praise & Gratitude           271
Account Issue                129
Performance Issue            100
Feature Complaint/Request     90
Technical Bug/Crash           72
General Request/Complaint     38
Monetization/Earning          23
Name: count, dtype: int64

In [4]:
dataset = df.content.values.tolist()

In [5]:
dataset

['sukaa',
 'ya bagus',
 'aku suka banget sama tiktok ini',
 'akun gua entah kenapa di Ben padalah gua kaga ngapa ngapain',
 'tolong diperbaiki',
 '😁',
 'cukup lumayan sih sama aplikasi ini , hanya saja banyak bug , kadang suka keluar sendiri dari aplikasi . terimakasih',
 'membagi inspirasi dan motivasi',
 'Sya kasi bintang 5 biyar lebih bagus lagi hai bosku kenpa titok ya sering elor terus adh aph dengan titok ya',
 'sangat bagus apk nya saya dapat uang juga dari tiktok',
 'mantap',
 'senang bgt',
 'aplikasi jelek',
 'bagus tapi akun ku di block...😭😭',
 'gaseru tiktok sampah saya gabisa join padahal tanggal lahir saya sudah benar',
 'Tiktok ini tidak ngeleg makanya saya download ini',
 'apk nya bagus saya suka, tapi diorang lain mempunyai potongan harga, sedang kan diakun saya ko tidak ada ya, tolong update kan akun saya agar mempunyai potongan harga.....terimakasih',
 'good',
 'tolong bug untuk postingan di atur ulang soalnya setiap muat postingan ngestag di 30% terus',
 'o aj sih😹',

In [6]:
%%time
model = ITER_DBSCAN(initial_distance=0.4, initial_minimum_samples=16, delta_distance=0.01, delta_minimum_samples=1, max_iteration=15, algorithm="IndoBERT", metric="euclidean")

CPU times: user 15 μs, sys: 1e+03 ns, total: 16 μs
Wall time: 21 μs


In [7]:
%%time
labels = model.fit_predict(dataset)

Loading HuggingFace model...
Model Loaded.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


CPU times: user 2min 33s, sys: 4.68 s, total: 2min 38s
Wall time: 42.8 s


In [8]:
df['cluster_ids'] = labels
df.cluster_ids.value_counts()

cluster_ids
-1     864
 1      42
 0      22
 2      19
 3       8
 4       8
 5       7
 6       6
 9       4
 7       4
 8       4
 10      3
 11      3
 12      3
 13      3
Name: count, dtype: int64

In [9]:
df.to_excel("result.xlsx", index=False)

In [10]:
evaluate_dataset = EvaluateDataset(filename=filepath, filetype='csv', text_column='content', 
                                   target_column='intent')

In [11]:
parameters = [
             {
    "distance": 0.4,           # Ditingkatkan agar cakupan lebih luas
    "minimum_samples":16, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    "algorithm": "IndoBERT",
    "metric": "euclidean"
},
{
    "distance": 0.4,           # Ditingkatkan agar cakupan lebih luas
    "minimum_samples":16, 
    "delta_distance":0.01, 
    "delta_minimum_samples":1, 
    "max_iteration":15,
    # "algorithm": "IndoBERT",
    # "metric": "euclidean"
}]

In [12]:
%%time
results = evaluate_dataset.evaulate_iter_dbscan(parameters)

Loading Tensorflow model....
Model Loaded.


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
100%|██████████| 2/2 [00:02<00:00,  1.24s/it]

CPU times: user 16.3 s, sys: 4.31 s, total: 20.7 s
Wall time: 13.8 s


In [13]:
result_df = pd.DataFrame.from_dict(results)
result_df

,distance,minimum_samples,delta_distance,delta_minimum_samples,max_iteration,algorithm,metric,time,percentage_labelled,clusters,...,homogeneity_score,completeness_score,normalized_mutual_info_score,adjusted_mutual_info_score,adjusted_rand_score,accuracy,precision,recall,f1,intents
0,0.4,16,0.01,1,15,IndoBERT,euclidean,1.35,13.3,14,...,0.05,0.24,0.08,0.07,-0.02,0.321,34.4,32.1,21.7,3
1,0.4,16,0.01,1,15,NaN,NaN,0.87,5.6,4,...,0.01,0.08,0.03,0.02,-0.01,0.242,12.7,24.2,13.7,2


In [14]:
result_df.to_csv("result_evaluation.csv", index=False)